In [6]:
#TODO make sure this renders in the github repo

# RMSNorm Root Mean Square Layer Normalization


**[Llama 1 Paper](https://arxiv.org/abs/2302.13971):** "We use the RMSNorm normalizing function, introduced by [Zhang and Sennrich (2019)](https://arxiv.org/pdf/1910.07467)." All Llama models use it.

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{ \frac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}} * \gamma$$

In [7]:
import EasyJupyter
import torch
import torch.nn as nn
import torch.nn.functional as F

In [8]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_config import BaseConfig

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, cfg: BaseConfig):
        """
        RMSNorm Root Mean Square Normalization
        """
        super().__init__()
        self.eps = cfg.rms_norm_eps

        # The learnable scaling parameter (gamma)
        self.weight = nn.Parameter(torch.ones(cfg.d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: The input tensor to be normalized, shape: (batch_size, context_len, d_model).
        """
        # Store the original data type
        orig_dtype = x.dtype
        
        x = x.to(torch.float32)

        # Calculate the variance
        variance = x.pow(2).mean(-1, keepdim=True)

        # Calculate the Root Mean Square and normalize
        x_normed = x * torch.rsqrt(variance + self.eps)

        # Case back to the original data type and apply the learnable weight
        return x_normed.to(orig_dtype) * self.weight